In [1]:
import numpy as np
import matplotlib.pyplot as plt

Condition number: error amplification
kappa = sigma_max / sigma_min.  Each power of 10 in kappa costs one digit
of accuracy.  A tiny change in b produces a large change in x when kappa >> 1.

In [3]:
A = np.array([[1.0, 1.0],
              [1.0, 1.0 + 1e-5]])
b = np.array([2.0, 2.0])
b_perturbed = b + [1e-5, 0.0]

x  = np.linalg.lstsq(A, b,           rcond=None)[0]
xp = np.linalg.lstsq(A, b_perturbed, rcond=None)[0]

print(f"kappa = {np.linalg.cond(A):.2e}")
print(f"relative error in b: {np.linalg.norm(b_perturbed - b) / np.linalg.norm(b):.2e}")
print(f"relative error in x: {np.linalg.norm(xp - x) / np.linalg.norm(x):.2e}")

kappa = 4.00e+05
relative error in b: 3.54e-06
relative error in x: 7.07e-01


Eigenvalues: the unit circle maps to an ellipse.
Eigenvectors are the axes of that ellipse; eigenvalues are the squared half-lengths.

In [4]:
A = np.array([[3.0, 1.0], [1.0, 3.0]])
vals, vecs = np.linalg.eigh(A)

t = np.linspace(0, 2 * np.pi, 300)
circle = np.stack([np.cos(t), np.sin(t)])
image  = A @ circle

fig, ax = plt.subplots()
ax.plot(*circle, label="unit circle")
ax.plot(*image,  label="image under A")
for i in range(2):
    ev = vecs[:, i] * vals[i]
    ax.plot([-ev[0], ev[0]], [-ev[1], ev[1]], "k--", lw=1)
ax.set_aspect("equal"); ax.legend(); ax.set_title("Eigenvectors = axes of the ellipse")
plt.show()

SVD geometry: A = U Sigma V^T decomposes any linear map into three steps.
V^T rotates the input, Sigma scales each axis, U rotates the output.

In [5]:
import numpy as np
A = np.array([[4.0, 1.0], [0.1, 2.0],[3.0, 1.0]])
U, s, Vt = np.linalg.svd(A)
print(U)
print(s)
print(Vt)



[[-0.78479493  0.17230334 -0.59532216]
 [-0.13972533 -0.98503584 -0.10090206]
 [-0.60379942  0.00399416  0.79712628]]
[5.23728329 1.89231704]
[[-0.94792477 -0.31849433]
 [ 0.31849433 -0.94792477]]


In [6]:
A = np.array([[4.0, 1.0], [0.1, 2.0]])
U, s, Vt = np.linalg.svd(A)

circle = np.stack([np.cos(t), np.sin(t)])
steps  = [circle, Vt @ circle, np.diag(s) @ Vt @ circle, U @ np.diag(s) @ Vt @ circle]
titles = ["input", "after V^T", "after Sigma", "after U  (= A @ input)"]

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, pts, title in zip(axes, steps, titles):
    ax.plot(*pts); ax.set_aspect("equal"); ax.set_title(title); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

Least squares: minimize ||Ax - b||^2.
The pseudoinverse x* = A^+ b is the minimum-norm solution.

In [7]:
rng = np.random.default_rng(0)
x_data = np.linspace(0, 5, 20)
y_data = 2.0 * x_data + 1.0 + rng.normal(0, 0.4, 20)

A_ls = np.column_stack([x_data, np.ones(20)])
c    = np.linalg.lstsq(A_ls, y_data, rcond=None)[0]
print(f"fit: y = {c[0]:.3f} x + {c[1]:.3f}  (true: y = 2x + 1)")

fig, ax = plt.subplots()
ax.scatter(x_data, y_data, s=20)
ax.plot(x_data, c[0] * x_data + c[1], label="fit")
ax.plot(x_data, 2 * x_data + 1, "k--", label="true")
for xi, yi, yp in zip(x_data, y_data, A_ls @ c):
    ax.plot([xi, xi], [yi, yp], "gray", lw=0.7, alpha=0.7)
ax.legend(); plt.show()

fit: y = 1.958 x + 1.032  (true: y = 2x + 1)


Low-rank approximation: the rank-k truncated SVD is the best rank-k matrix
in the Frobenius norm (Eckart-Young theorem).

In [ ]:
rng = np.random.default_rng(1)
A = np.outer([1, 2, 3, 4], [4, 3, 2, 1]) + rng.normal(0, 0.3, (4, 4))
# play around with a differnt A to see how the singular values and low-rank approximations change
# A = np.array([[4.0, 1.0], [0.1, 2.0],[3.0, 1.0],[1.0, 4.0]])
U, s, Vt = np.linalg.svd(A, full_matrices=False)

print(f"singular values: {s.round(3)}")
fig, axes = plt.subplots(1, 4, figsize=(10, 2.5))
for ax, k in zip(axes, range(1, 5)):
    Ak = U[:, :k] @ np.diag(s[:k]) @ Vt[:k]
    ax.imshow(Ak, cmap="coolwarm", vmin=A.min(), vmax=A.max())
    ax.set_title(f"rank {k}  err={np.linalg.norm(A - Ak):.2f}"); ax.axis("off")
plt.tight_layout(); plt.show()

singular values: [5.948 3.553]


Covariance propagation: Sigma_y = A Sigma_x A^T.
The linear map A stretches and rotates the uncertainty ellipse.
This is the Kalman filter predict step.

In [11]:
Sigma_x = np.array([[4.0, 2.0], [2.0, 1.0]])
A       = np.array([[1.0, 0.0], [1.0, 1.0]])
Sigma_y = A @ Sigma_x @ A.T

def ellipse(S, n=2):
    vals, vecs = np.linalg.eigh(S)
    c = np.stack([np.cos(t), np.sin(t)])
    return vecs @ np.diag(np.sqrt(vals) * n) @ c

fig, ax = plt.subplots()
ax.plot(*ellipse(Sigma_x), label="Sigma_x")
ax.plot(*ellipse(Sigma_y), label="Sigma_y = A Sigma_x A^T")
ax.set_aspect("equal"); ax.legend(); ax.grid(True, alpha=0.3)
ax.set_title("Covariance propagation (2-sigma ellipses)"); plt.show()